# **Cross-Species and Cross-Condition Meta-Analysis of Microglial Dynamics**
## Notebook 01: Dataset Preprocessing, Quality Control, Leiden Clustering, Marker Gene Identification, QBR, and Pseudobulk Processing

**Project:** Cross-Species and Cross-Condition Meta-Analysis of Microglial Alterations in Brain Aging and Neurodegeneration  
**Environment:** Optimized for Google Colab execution. Execution in alternative local environments (e.g., standard Jupyter Notebooks) may require minor path and dependency modifications.

### Pipeline Overview
This notebook contains the official computational pipeline for the initial processing and downstream analysis of **Dataset 1 (Martirosyan et al., 2024)**. It implements a rigorous data harmonization pipeline spanning:
* Raw UMI count acquisition and loading
* Cell- and gene-level quality control (QC) filtering
* Doublet detection and removal via Scrublet
* Automated cell-type annotation using CellTypist
* High-confidence myeloid cell selection
* Batch integration via BBKNN

Following preprocessing, the notebook outlines the core analytical frameworks used to characterize microglial dynamics: Leiden clustering, marker gene identification, quasi-binomial logistic regression (QBR) for condition enrichment, and pseudobulk differential expression profiling.

*Note: While independent datasets analyzed in this study may differ in technical specifics (e.g., repository sourcing, macaque-specific annotation models, or stringent multi-comparison corrections), they uniformly adhere to the execution logic established in this reference pipeline.*

## **Section 1: Environment Setup & Dependencies**

To ensure the pipeline runs reliably and reproduces the exact same results, we first set up our computational environment. We configure the `SciPy` array backend for consistent calculations and suppress minor package updates/warnings to keep the output logs clean.

In [ ]:
import os
os.environ["SCIPY_ARRAY_API"] = "1"

In [ ]:
!pip install scanpy[leiden] scrublet bbknn harmonypy scanorama celltypist mygene

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import the libraries

import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import seaborn as sns
import matplotlib.pyplot as plt
import anndata as ad
from tqdm import tqdm

import mygene
import bbknn

import celltypist
from celltypist import models
from celltypist import annotate

In [ ]:
# Set visualization parameters

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['figure.dpi'] = 300

# **Section 2: Data Sourcing, Downloading, and Loading**

In this section, we download the raw single-nucleus RNA-sequencing (snRNA-seq) data directly from the NCBI Gene Expression Omnibus (GEO). The script automatically sets up folders for each donor, downloads the required matrix components, and loads them into Scanpy AnnData objects for downstream processing.

In [ ]:
# Automated download and file renaming of 10X matrices from NCBI GEO FTP

import os

# List of all GSM IDs and their corresponding sample suffixes
samples = [
    ("GSM7792123", "s_0096"), ("GSM7792124", "s_0098"), ("GSM7792125", "s_0100"),
    ("GSM7792126", "s_0102"), ("GSM7792127", "s_0103"), ("GSM7792128", "s_0104"),
    ("GSM7792129", "s_0105"), ("GSM7792130", "s_0107"), ("GSM7792131", "s_0109"),
    ("GSM7792132", "s_0110"), ("GSM7792133", "s_0111"), ("GSM7792134", "s_0116"),
    ("GSM7792135", "s_0118"), ("GSM7792136", "s_0119"), ("GSM7792137", "s_0120"),
    ("GSM7792138", "s_0127"), ("GSM7792139", "s_0128"), ("GSM7792140", "s_0129"),
    ("GSM7792141", "s_0130"), ("GSM7792142", "s_0131"), ("GSM7792143", "s_0142"),
    ("GSM7792144", "s_0147"), ("GSM7792145", "s_0151"), ("GSM7792146", "s_0152"),
    ("GSM7792147", "s_0153"), ("GSM7792148", "s_0154"), ("GSM7792149", "s_0158"),
    ("GSM7792150", "s_0159"), ("GSM7792151", "s_0165"),
]

print(f"📥 Total samples to process: {len(samples)}")

for gsm_id, sample_id in samples:
    print(f"\n--- Downloading {gsm_id} ({sample_id}) ---")

    # 1. Create a local target directory
    os.makedirs(sample_id, exist_ok=True)

    # 2. Build target URLs and local filenames
    base_url = f"https://ftp.ncbi.nlm.nih.gov/geo/samples/{gsm_id[:7]}nnn/{gsm_id}/suppl/"
    files = ["barcodes.tsv.gz", "features.tsv.gz", "matrix.mtx.gz"]
    prefixes = [f"{gsm_id}_{sample_id}"] * 3

    # 3. Fetch and clean up data layout
    for prefix, fname in zip(prefixes, files):
        src_url = base_url + prefix + "_" + fname
        dst_path = os.path.join(sample_id, fname)

        # Download file using curl
        !curl -s -O {src_url}

        # Move and rename to match standard 10x structure
        !mv {prefix}_{fname} {dst_path}

print("\n✅ All sample files downloaded and structured successfully!")

In [ ]:
# Load standard 10X sparse matrices into memory as Scanpy AnnData objects

adatas = {}

for _, sample_id in samples:
    print(f"Reading matrix for {sample_id}...")

    # Load 10x mtx files using gene symbols as var indexes
    adata = sc.read_10x_mtx(sample_id, var_names='gene_symbols', cache=True)

    # Store origin sample ID in metadata slot
    adata.obs['sample'] = sample_id
    adatas[sample_id] = adata

print(f"\n✅ Successfully loaded {len(adatas)} samples into memory.")

In [ ]:
# A typical object has the following structure

adatas['s_0118']

# **Section 3: Quality Control Metrics Calculation**

In this section, we apply strict filtering thresholds to remove low-quality cells, dying cells, and doublets. We filter cells based on minimum/maximum UMI counts and total detected genes. We also exclude cells with high mitochondrial or hemoglobin expression, and use Scrublet to identify and remove potential doublets.

In [ ]:
# Identify mitochondrial/hemoglobin genes and compute core single-cell QC metrics

for sn in adatas.keys():
    # Tag mitochondrial genes (typically starting with MT-)
    mt_mask = adatas[sn].var.index.astype(str).str.startswith('MT-')
    adatas[sn].var['mt'] = mt_mask.astype(bool)

    # Tag hemoglobin genes (HB* genes, excluding pseudogenes like HBP)
    hb_mask = adatas[sn].var.index.astype(str).str.contains("^HB[^(P)]")
    adatas[sn].var['hb'] = hb_mask.astype(bool)

    # Calculate standard QC metrics using Scanpy
    sc.pp.calculate_qc_metrics(
        adatas[sn],
        qc_vars=['mt', 'hb'],
        percent_top=None,
        log1p=False,
        inplace=True
    )

print("✅ QC metrics calculated for all samples.")

In [ ]:
# Define and execute a diagnostic plotting function to visualize key QC metric distributions across all samples

def plot_qc(adatas, features):
    for sn in adatas.keys():
        fig, axs = plt.subplots(1, len(features), figsize=(16, 4))
        fig.suptitle(f'Sample: {sn}', fontsize=16)

        for j, feat in enumerate(features):
            ax = axs[j]
            data = adatas[sn].obs[feat].dropna()
            if len(data) > 0:
                sns.histplot(data, ax=ax, log_scale=[True, False])
            else:
                ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center')
            ax.set_xlabel(feat)
            ax.set_ylabel('Count')

        plt.tight_layout()
        plt.show()

features = ["total_counts", "n_genes_by_counts", "pct_counts_mt", "pct_counts_hb"]
plot_qc(adatas, features)

In [ ]:
# Define a strict QC filtering pipeline to remove low-quality cells and doublets

def manual_filter_adata(adata):
    # Filter cells by UMI counts and total detected genes
    sc.pp.filter_cells(adata, min_counts=500)
    sc.pp.filter_cells(adata, max_counts=10000)
    sc.pp.filter_cells(adata, max_genes=6000)

    # Filter genes to ensure they are expressed in at least 5 cells
    sc.pp.filter_genes(adata, min_cells=5)

    # Exclude cells with high mitochondrial or hemoglobin percentages
    adata = adata[adata.obs.pct_counts_mt < 10, :].copy()
    adata = adata[adata.obs.pct_counts_hb < 1, :].copy()

    # Run Scrublet to detect and remove doublets
    sc.pp.scrublet(adata, batch_key=None, random_state=42)
    if 'predicted_doublet' in adata.obs:
        adata = adata[~adata.obs['predicted_doublet'], :].copy()

    return adata

In [ ]:
# Apply the custom filtering pipeline to all loaded single-nucleus samples

adatas_manual_filt = {}
for sn, adata in adatas.items():
    print(f"Filtering sample {sn}...")
    adatas_manual_filt[sn] = manual_filter_adata(adata)

print(f"\n✅ Filtered {len(adatas_manual_filt)} out of {len(adatas)} samples.")

In [ ]:
# Generate a summary report comparing cell counts before and after quality control filtering

print("📊 Filtration Summary Report:")
print(f"{'Sample':<25} {'Before':<10} {'After':<10} {'% Kept':<8}")
print("-" * 55)

for sn in adatas.keys():
    n_before = adatas[sn].n_obs
    # Use get() with a fallback value to safely check if the sample exists in filtered data
    n_after = adatas_manual_filt[sn].n_obs if sn in adatas_manual_filt else 0
    pct_kept = (n_after / n_before * 100) if n_before > 0 else 0
    print(f"{sn:<25} {n_before:<10} {n_after:<10} {pct_kept:<8.1f}")

To prevent data loss from Google Colab session timeouts, we save the filtered AnnData objects directly to Google Drive as compressed `.h5ad` files. We also provide a script to reload these processed matrices back into memory for downstream analysis.


In [ ]:
# Export individual filtered samples as .h5ad files to Google Drive

# 1. Define output directory path
save_dir = '/content/drive/MyDrive/Martirosyan_microglia_Parkinson_dataset/'
os.makedirs(save_dir, exist_ok=True)

# 2. Loop through and export each filtered AnnData object
print("🚀 Starting to save filtered samples to Google Drive...\n")

for sample_name, adata in adatas_manual_filt.items():
    # Construct individual file path
    file_path = os.path.join(save_dir, f"{sample_name}.h5ad")

    # Write object to disk
    adata.write(file_path)
    print(f"✅ Saved {sample_name}: {adata.n_obs} nuclei, {adata.n_vars} genes")

print(f"\n🎉 All filtered samples successfully stored on Google Drive in:\n{save_dir}")

In [ ]:
# Discover and reload all processed sample files from Google Drive

# Define input directory path
load_dir = '/content/drive/MyDrive/Martirosyan_microglia_Parkinson_dataset/'

# Retrieve a clean list of all processed sample IDs from available .h5ad files
sample_names = [
    f.replace('.h5ad', '')
    for f in os.listdir(load_dir)
    if f.endswith('.h5ad') and f.startswith('s_0')
]

# Initialize storage dictionary
adatas_manual_filt = {}

# Load each file sequentially
for sample_name in sample_names:
    file_path = os.path.join(load_dir, f"{sample_name}.h5ad")
    adata = sc.read(file_path)
    adatas_manual_filt[sample_name] = adata
    print(f"📥 Loaded {sample_name}: {adata.n_obs} nuclei, {adata.n_vars} genes")

print(f"\n✅ Successfully reloaded {len(adatas_manual_filt)} processed samples into the environment.")

# **Section 4: Data Normalization and Log Transformation**

In this section, we prepare the filtered count matrices for downstream statistical analysis. We scale the raw UMI counts within each cell to a standard library size (10,000 counts per cell) to account for differences in sequencing depth, and then apply a log1p transformation to stabilize variance.


In [ ]:
# Cell purpose: Perform total count normalization and log1p transformation for all filtered samples
for sample_name, adata in adatas_manual_filt.items():

    # Save a backup of the raw UMI counts in the "counts" layer
    adata.layers["counts"] = adata.X.copy()

    # Normalize each cell to a target sum of 10,000 UMI counts
    sc.pp.normalize_total(adata, target_sum=1e4)

    # Apply log1p transformation: log(1 + x)
    sc.pp.log1p(adata)

    print(f"✅ {sample_name}: normalization and log1p transformations complete.")

In [ ]:
# Extract and plot raw UMI count distributions for select genes using violin and strip plots

import scipy

# Select a representative sample for validation
sample_name = 's_0098'
adata = adatas_manual_filt[sample_name]

# Target genes for analysis (Ensembl IDs)
gene_ids = [
    "ENSG00000136235",  # GPNMB
    "ENSG00000170345",  # FOS
    "ENSG00000116251"   # RPL22
]

# Plot colors
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

# Prepare data storage
data_list = []

# Verify that the 'gene_ids' column exists in adata.var metadata
if 'gene_ids' not in adata.var.columns:
    raise ValueError("The 'gene_ids' column was not found in adata.var. Please check your data structure.")

# Create an Ensembl ID to gene symbol mapping dictionary
ensembl_to_symbol = pd.Series(adata.var.index, index=adata.var['gene_ids']).to_dict()

for gene_id in gene_ids:
    if gene_id not in ensembl_to_symbol:
        print(f"❌ Ensembl ID {gene_id} not found in adata.var['gene_ids']")
        continue

    gene_symbol = ensembl_to_symbol[gene_id]

    # Find the gene by its symbol since the X matrix features use gene symbols
    if gene_symbol not in adata.var_names:
        print(f"❌ Gene symbol {gene_symbol} not found in adata.var_names")
        continue

    # Safely extract raw expression values, handling both sparse and dense matrices
    X_gene = adata[:, gene_symbol].X
    expr = X_gene.toarray().flatten() if scipy.sparse.issparse(X_gene) else X_gene.flatten()

    df_gene = pd.DataFrame({
        'Expression': expr,
        'Gene': f'{gene_symbol}\n({gene_id})'
    })
    data_list.append(df_gene)

# Check data availability and generate the plot
if not data_list:
    print("⚠️ No matching genes found — plot generation skipped.")
else:
    plot_data = pd.concat(data_list, ignore_index=True)

    # Build the combined violin and strip plot layout
    plt.figure(figsize=(12, 6))
    ax = sns.violinplot(
        data=plot_data,
        x='Gene',
        y='Expression',
        palette=colors,
        cut=0,
        inner=None,
        linewidth=1.2,
        width=0.8
    )

    sns.stripplot(
        data=plot_data,
        x='Gene',
        y='Expression',
        color='black',
        alpha=0.7,
        s=3,
        jitter=True,
        edgecolor='gray',
        linewidth=0.5,
        ax=ax
    )

    # Apply clean formatting and axis labels
    plt.title(f"Distribution of Raw UMI Counts\nSample: {sample_name}", fontsize=16, pad=20)
    plt.ylabel("UMI Counts (raw)", fontsize=12)
    plt.xlabel("Gene", fontsize=12)
    plt.xticks(fontsize=11)
    plt.yticks(fontsize=10)

    # Set display bounds and add grid background
    max_y = plot_data['Expression'].max()
    plt.ylim(-0.5, max_y + 0.5)
    plt.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.7)

    plt.tight_layout()
    plt.show()

# **Section 5: Automated Cell Type Annotation via CellTypist & Myeloid Filtering**

In this section, we use CellTypist with a pre-trained adult human prefrontal cortex model to automatically annotate cell types across all samples. After annotation, we filter the dataset to keep only the myeloid cell populations (microglia and macrophages) for our downstream analysis.


In [ ]:
# Download and load the pre-trained CellTypist model architecture for brain tissue annotation

models.download_models(
    force_update=True,
    model=["Adult_Human_PrefrontalCortex.pkl"] # provide model name
)

# Load model
model = models.Model.load(model="Adult_Human_PrefrontalCortex.pkl")

In [ ]:
# Run CellTypist annotation across all filtered samples and transfer labels to metadata

adatas_annotated = {}

for sample_name, adata in adatas_manual_filt.items():
    # Create a copy and ensure gene symbols are set as index without duplicates
    adata_cp = adata.copy()
    adata_cp.var_names = adata_cp.var_names.astype(str)
    adata_cp = adata_cp[:, ~adata_cp.var_names.duplicated(keep='first')].copy()

    # Run automated cell type annotation with majority voting
    prediction = annotate(adata_cp, model=model, majority_voting=True)

    # Transfer high-confidence annotation labels back to the main object metadata
    adata.obs['predicted_cell_type'] = prediction.predicted_labels['predicted_labels']
    adata.obs['majority_voting'] = prediction.predicted_labels['majority_voting']

    adatas_annotated[sample_name] = adata
    print(f"✅ {sample_name}: cell type annotation complete.")

In [ ]:
# Filter annotated samples for specific myeloid lineages and concatenate them into a single object

# Define the target microglial and macrophage cell types from the CellTypist model
target_types = [
    'Macro F13A1 COLEC12',
    'Micro P2RY12 APBB1IP',
    'Micro P2RY12 CCL3',
    'Micro P2RY12 GLDN',
    'Myeloid LSP1 LYZ'
]

# Initialize list to store filtered subset objects
microglia_list = []

print("🔍 Filtering myeloid cell populations across samples:\n")
print(f"{'Sample':<25} {'Total Nuclei':<14} {'Myeloid Nuclei':<18} {'% Kept'}")
print("-" * 70)

for sample_name, adata in adatas_annotated.items():

    # Create a boolean mask to filter for target cell types
    mask = adata.obs['majority_voting'].isin(target_types)
    subset = adata[mask].copy()

    # Append the filtered subset to our collection list
    microglia_list.append(subset)

    # Compute profiling counts and percentages for reporting
    total_cells = adata.n_obs
    filtered_cells = subset.n_obs
    pct_kept = (filtered_cells / total_cells * 100) if total_cells > 0 else 0
    print(f"{sample_name:<25} {total_cells:<14} {filtered_cells:<18} {pct_kept:<.1f}%")

# Concatenate all filtered individual subsets into one master object
combined = sc.concat(microglia_list, join='outer')

print("-" * 70)
print(f"✅ Success: Combined a total of {combined.n_obs} myeloid nuclei across {len(microglia_list)} samples.")

To protect our workflow from Google Colab session timeouts and unexpected runtime crashes, we save the unified, annotated myeloid matrix (`combined`) to Google Drive as an `.h5ad` file. This acts as a critical checkpoint, allowing us to immediately resume downstream analyses—such as batch integration, clustering, and differential expression—without wasting time re-running the heavy preprocessing and CellTypist steps.

In [ ]:
# Export the combined myeloid dataset to Google Drive as a checkpoint file

save_path = '/content/drive/MyDrive/Martirosyan_microglia_Parkinson_dataset/combined_microglia_macrophages.h5ad'

# Save the combined object to disk
combined.write(save_path)

print(f"✅ Saved successfully: {combined.n_obs} nuclei × {combined.n_vars} genes")

In [ ]:
# Reload the verified combined myeloid dataset back into memory

load_path = '/content/drive/MyDrive/Martirosyan_microglia_Parkinson_dataset/combined_microglia_macrophages.h5ad'

# Load the file back into an AnnData object
combined = sc.read(load_path)

print(f"📥 Combined dataset successfully loaded into memory:")
print(f"   {combined.n_obs} nuclei | {combined.n_vars} genes")
print(f"   Included cell types: {combined.obs['majority_voting'].unique().tolist()}")

# **Section 6: Metadata, HVG, PCA, BBKNN, Leiden, UMAP**

In this section, we transition from individual samples to a unified downstream analysis. We add the metadata, select highly variable genes (HVGs) to capture biological variance, and run Principal Component Analysis (PCA) for dimensionality reduction. To account for batch effects across different donors, we apply BBKNN during the neighbor graph construction before performing final Leiden clustering and UMAP visualization.

In [ ]:
# Map external donor clinical metadata (Diagnosis, Age, Sex) to the combined AnnData object

# Define metadata mapping matching individual sample suffixes
metadata_dict = {
    'sample': [
        's_0096', 's_0098', 's_0100', 's_0102', 's_0103', 's_0104', 's_0105', 's_0107',
        's_0109', 's_0110', 's_0111', 's_0116', 's_0118', 's_0119', 's_0120', 's_0127',
        's_0128', 's_0129', 's_0130', 's_0131', 's_0142', 's_0147', 's_0151', 's_0152',
        's_0153', 's_0154', 's_0158', 's_0159', 's_0165'
    ],
    'Clinical diagnosis': [
        "Parkinson's", "Parkinson's", "Parkinson's", "Parkinson's", "Parkinson's", "Parkinson's", "Parkinson's", "Parkinson's",
        "Parkinson's", "Parkinson's", "Parkinson's", "Parkinson's", "Parkinson's", "Parkinson's", "Parkinson's", "Control",
        "Control", "Control", "Control", "Control", "Control", "Control", "Control", "Control",
        "Control", "Control", "Control", "Control", "Control"
    ],
    'Age': [
        90, 99, 89, 57, 68, 81, 78, 82,
        98, 82, 80, 77, 85, 84, 85, 65,
        81, 86, 61, 65, 62, 30, 77, 93,
        91, 93, 89, 73, 91
    ],
    'Sex': [
        'female', 'male', 'male', 'female', 'male', 'male', 'male', 'male',
        'male', 'male', 'male', 'male', 'female', 'male', 'female', 'male',
        'female', 'male', 'male', 'male', 'male', 'female', 'female', 'female',
        'male', 'male', 'male', 'female', 'male'
    ]
}

# Create DataFrame and index it by sample column for alignment
metadata_df = pd.DataFrame(metadata_dict)
metadata_df = metadata_df.set_index('sample')[['Clinical diagnosis', 'Age', 'Sex']]

# Map annotations into original combined.obs slot using sample ID as key
combined.obs = combined.obs.join(metadata_df, on='sample')

# Final verification report
print("✅ Successfully appended metadata features:", ['Clinical diagnosis', 'Age', 'Sex'])
print("\nExample metadata preview:")
print(combined.obs[['sample', 'Clinical diagnosis', 'Age', 'Sex']].head())

In [ ]:
# Identify highly variable genes (HVGs) using Pearson residuals, accounting for batch effects

sc.experimental.pp.highly_variable_genes(
    combined,
    n_top_genes=3000,
    flavor='pearson_residuals',
    batch_key='sample'
)

print(f"✅ Found {combined.var['highly_variable'].sum()} highly variable genes (HVGs).")

In [ ]:
# Perform Principal Component Analysis (PCA) restricted to highly variable genes

sc.tl.pca(
    combined,
    use_highly_variable=True,
    svd_solver='arpack',
    random_state=42
)

print("✅ PCA dimensionality reduction completed.")

In [ ]:
# Build a batch-corrected neighbor graph using the BBKNN algorithm across donor samples

sce.pp.bbknn(
    combined,
    batch_key='sample',
    use_rep='X_pca',
    n_pcs=40
)

print("✅ Batch correction and neighbor graph construction via BBKNN completed.")

In [ ]:
# Perform UMAP

sc.tl.umap(combined)

In [ ]:
# Compute Leiden clustering at resolution 2.0

sc.tl.leiden(
    combined,
    resolution=2.0,
    key_added='leiden_res_2.0',
    random_state=42
)

In [ ]:
# Generate publication-ready UMAP plot

sc.pl.umap(
    combined,
    color='leiden_res_2.0',
    title='Leiden Clustering (Resolution = 2.0)',
    frameon=False,
    size=10
)

In [ ]:
# Vusualize residual batch effect

sc.pl.umap(
    combined,
    color='sample',
    title='Batch Effect',
    frameon=False,
    size=10
)

In [ ]:
# Export the final clustered myeloid dataset to Google Drive as a checkpoint file

save_path = '/content/drive/MyDrive/Martirosyan_microglia_Parkinson_dataset/combined_clusters.h5ad'

# Save the clustered object to disk
combined.write(save_path)

print(f"✅ Success: Clustered dataset saved to {save_path}")

In [ ]:
# Reload the verified clustered myeloid dataset back into memory for downstream analysis

load_path = '/content/drive/MyDrive/Martirosyan_microglia_Parkinson_dataset/combined_clusters.h5ad'

# Load the file into a new analytical object variable
combined = sc.read(load_path)

print(f"📥 Dataset successfully reloaded:")
print(f"   {combined.n_obs} nuclei | {combined.n_vars} genes")
print(f"   Available clusters: {len(combined.obs['leiden_res_2.0'].unique())} groups identified")

In [ ]:
combined

# **Section 7: Marker Genes**

In this section, we identify highly specific marker genes for each of our Leiden clusters. We perform differential expression (DE) testing using the Wilcoxon rank-sum test and apply Benjamini-Hochberg multi-comparison correction to discover high-confidence gene signatures that define each myeloid subpopulation.

In [ ]:
# Differential expression analysis

sc.tl.rank_genes_groups(
    combined,
    groupby='leiden_res_2.0',
    method='wilcoxon',
    corr_method='benjamini-hochberg',
    use_raw=False
)

In [ ]:
# Generate individual diagnostic dotplots displaying top 10 marker genes for each Leiden cluster

# 1. Prepare data object for plotting by deduplicating gene symbols
adata_plot = combined.copy()
adata_plot.var_names = adata_plot.var_names.astype(str)
adata_plot = adata_plot[:, ~adata_plot.var_names.duplicated(keep='first')].copy()

# 2. Extract unique identified clusters
clusters = combined.obs['leiden_res_2.0'].cat.categories

# 3. Iterate through each cluster and generate marker gene dotplots
for cluster in clusters:
    # Retrieve top 10 marker genes for the current cluster
    top_symbols = list(combined.uns['rank_genes_groups']['names'][cluster][:10])

    # Filter to ensure genes exist in the prepared plot object
    top_symbols = [g for g in top_symbols if g in adata_plot.var_names]

    if not top_symbols:
        print(f"⚠️ Warning: No valid marker genes found for Cluster {cluster}")
        continue

    # Plot expression and cell fraction across all clusters
    sc.pl.dotplot(
        adata_plot,
        var_names=top_symbols,
        groupby='leiden_res_2.0',
        use_raw=False,
        standard_scale='var',
        title=f'Cluster {cluster} — Top 10 Markers',
        figsize=(6, 4),
        dendrogram=False,
        cmap='viridis',
        show=True
    )

In [ ]:
# Define a function for customized dotplot generation

def plot_custom_dotplot(adata_plot, gene_list, groupby='leiden_res_2.0', title="Custom Gene Set"):
    """
    Generates a standardized dotplot for a custom list of genes and forces gene names to be italicized.
    """
    # Filter the list to include only genes present in the dataset matrix
    genes = [g for g in gene_list if g in adata_plot.var_names]

    # Call the Scanpy dotplot engine. show=False is mandatory for downstream axis modification.
    dp = sc.pl.dotplot(
        adata_plot,
        var_names=genes,
        groupby=groupby,
        use_raw=False,
        standard_scale='var',
        title=title,
        figsize=(12, 4),
        dendrogram=False,
        cmap='viridis',
        show=False
    )

    plt.show()

In [ ]:
# Plot custom myeloid gene sets

target_genes = [
    # Macrophage markers
    "F13A1", "CD163", "CD163L1", "COLEC12",

    # Uncategorized transition genes
    "GRID2", "CCDC26",

    # Homeostatic
    "P2RY12", "CX3CR1", "SORL1", "MEF2A", "ITPR2", "FRMD4A", "ELMO1", "ANKRD44", "SRGAP2",

    # GPNMB+ signature
    "GPNMB", "MITF", "PPARG", "PTPRG", "MYO1E", "CPM", "KCNMA1", "ATG7", "IQGAP2",
    "STARD13", "DPYD", "LRRK2", "FOXP1", "APOE", "SERPINE1",

    # Inflammation & Related
    "SPP1", "TMEM163", "MSR1", "SLC11A1", "CD83", "IL1B", "IRAK2",

    # Ribosomal & Immune
    "RPL32", "RPS19", "C1QB", "FTH1", "TMSB10", "HLA-B",

    # HSPs
    "HSPH1", "DNAJB1", "HSP90AA1"
]

plot_custom_dotplot(adata_plot, target_genes, title="Myeloid Lineage & Activation Markers")

# **Section 8: Quasi-Binomial Logistic Regression**

In this section, we evaluate which myeloid clusters are significantly enriched or depleted in Parkinson's Disease (PD) compared to healthy controls.

First, we use a Quasi-Binomial Logistic Regression framework (via Generalized Linear Models with a Binomial family and robust standard errors) to model cell type proportions while accounting for donor-to-donor variability. Then, we visualize these statistical outputs using a diagnostic bubble plot, where the Y-axis shows the Log Odds Ratio (PD vs Control), the color indicates FDR significance, and the bubble size reflects the total number of cells in each cluster.

In [ ]:
# Perform Quasi-Binomial Regression to evaluate cluster enrichment between clinical conditions

import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

# ------------------------------------------------------------
# Part 1: Data Preparation
# ------------------------------------------------------------
# Define target metadata columns required for regression analysis
required_cols = ['sample', 'Clinical diagnosis', 'leiden_res_2.0']
if not all(col in combined.obs.columns for col in required_cols):
    raise ValueError(f"Missing columns in combined.obs: {required_cols}")

metadata = combined.obs[required_cols].copy()

# Aggregate cell counts per sample and cluster
cluster_counts = metadata.groupby(['sample', 'leiden_res_2.0'], observed=False).size().reset_index(name='cluster_count')
total_counts = metadata.groupby('sample', observed=False).size().reset_index(name='total_sample_count')
agg_data = pd.merge(cluster_counts, total_counts, on='sample')

# Merge condition labels into the aggregated count framework
condition_info = metadata[['sample', 'Clinical diagnosis']].drop_duplicates()
agg_data = pd.merge(agg_data, condition_info, on='sample')

# Calculate the reciprocal count slot (cells not in the current cluster)
agg_data['non_cluster_count'] = agg_data['total_sample_count'] - agg_data['cluster_count']
agg_data = agg_data[agg_data['cluster_count'] > 0].copy()

# Enforce condition variable as categorical factor
agg_data['Clinical diagnosis'] = agg_data['Clinical diagnosis'].astype('category')

# FIXED: Replaced 'condition' key error with 'Clinical diagnosis' column reference
if len(agg_data['Clinical diagnosis'].cat.categories) != 2:
    raise ValueError("The 'Clinical diagnosis' column must contain exactly two categories (e.g., 'Parkinson's' and 'Control').")

categories = agg_data['Clinical diagnosis'].cat.categories.tolist()
ref_group, test_group = categories[0], categories[1]

# Calculate total baseline cellularity per condition for percentage scaling
total_cells_by_condition = metadata['Clinical diagnosis'].value_counts()
total_ref = total_cells_by_condition.get(ref_group, 0)
total_test = total_cells_by_condition.get(test_group, 0)

# ------------------------------------------------------------
# Part 2: Cluster-by-Cluster Generalized Linear Modeling
# ------------------------------------------------------------
results = []
unique_clusters = agg_data['leiden_res_2.0'].unique()
MIN_SAMPLES_PER_GROUP_FOR_RELIABLE = 3

for cluster_id in unique_clusters:
    cluster_data = agg_data[agg_data['leiden_res_2.0'] == cluster_id].copy()
    n_samples = len(cluster_data)

    cells_by_condition = cluster_data.groupby('Clinical diagnosis', observed=False)['cluster_count'].sum()
    cells_ref = cells_by_condition.get(ref_group, 0)
    cells_test = cells_by_condition.get(test_group, 0)

    pct_ref = 100 * cells_ref / total_ref if total_ref > 0 else 0.0
    pct_test = 100 * cells_test / total_test if total_test > 0 else 0.0

    # Compute proportional dispersion indices
    cluster_data['prop'] = cluster_data['cluster_count'] / cluster_data['total_sample_count']
    props_ref = cluster_data[cluster_data['Clinical diagnosis'] == ref_group]['prop']
    props_test = cluster_data[cluster_data['Clinical diagnosis'] == test_group]['prop']

    mean_prop_ref = props_ref.mean() if len(props_ref) > 0 else np.nan
    mean_prop_test = props_test.mean() if len(props_test) > 0 else np.nan

    def calc_cv(series):
        if len(series) == 0 or series.mean() == 0:
            return np.nan
        return series.std(ddof=1) / series.mean()

    cv_ref = calc_cv(props_ref)
    cv_test = calc_cv(props_test)

    # Evaluate statistical inference quality based on replication depth
    counts_by_group = cluster_data['Clinical diagnosis'].value_counts()
    n_ref = counts_by_group.get(ref_group, 0)
    n_test = counts_by_group.get(test_group, 0)

    if n_ref == 0 or n_test == 0:
        inference_quality = "undetermined"
        coef = np.nan
        p_value = np.nan
    elif n_ref < MIN_SAMPLES_PER_GROUP_FOR_RELIABLE or n_test < MIN_SAMPLES_PER_GROUP_FOR_RELIABLE:
        inference_quality = "low_replication"
        coef = np.nan
        p_value = np.nan
    else:
        inference_quality = "reliable"
        try:
            y = cluster_data['cluster_count'] / cluster_data['total_sample_count']
            weights = cluster_data['total_sample_count']
            X = pd.get_dummies(cluster_data['Clinical diagnosis'], drop_first=True, dtype=float)
            X = sm.add_constant(X)

            target_col = [col for col in X.columns if col != 'const']
            if len(target_col) == 1:
                target_col = target_col[0]
                model = sm.GLM(y, X, family=sm.families.Binomial(), var_weights=weights)
                fit = model.fit(cov_type='HC0')  # Quasi-binomial formulation with robust standard errors
                coef = fit.params[target_col]
                p_value = fit.pvalues[target_col]
            else:
                coef, p_value = np.nan, np.nan
        except Exception:
            coef, p_value = np.nan, np.nan

    if pd.isna(coef):
        enriched = "undetermined"
    elif coef > 0:
        enriched = test_group
    else:
        enriched = ref_group

    results.append({
        'cluster': cluster_id,
        'n_samples': n_samples,
        f'n_samples_in_{ref_group}': n_ref,
        f'n_samples_in_{test_group}': n_test,
        'inference_quality': inference_quality,
        'total_cells_in_cluster': cells_ref + cells_test,
        f'cells_in_{ref_group}': cells_ref,
        f'cells_in_{test_group}': cells_test,
        f'pct_in_{ref_group}': pct_ref,
        f'pct_in_{test_group}': pct_test,
        f'mean_prop_in_{ref_group}': mean_prop_ref,
        f'mean_prop_in_{test_group}': mean_prop_test,
        f'cv_in_{ref_group}': cv_ref,
        f'cv_in_{test_group}': cv_test,
        'enriched_in': enriched,
        'coef': coef,
        'odds_ratio': np.exp(coef) if not pd.isna(coef) else np.nan,
        'p_value': p_value
    })

# ------------------------------------------------------------
# Part 3: Multiple Comparison Correction and Export
# ------------------------------------------------------------
results_df = pd.DataFrame(results)
valid_p = results_df['p_value'].notna()

if valid_p.any():
    _, p_adj, _, _ = multipletests(results_df.loc[valid_p, 'p_value'], method='fdr_bh')
    results_df.loc[valid_p, 'p_adjusted'] = p_adj
else:
    results_df['p_adjusted'] = np.nan

# Sort outputs systematically by statistical reliability
sort_order = {"reliable": 0, "low_replication": 1, "undetermined": 2}
results_df['sort_key'] = results_df['inference_quality'].map(sort_order).fillna(3)
results_df = results_df.sort_values(by=['sort_key', 'p_adjusted', 'p_value'], na_position='last').reset_index(drop=True)
results_df = results_df.drop(columns=['sort_key'])

# Export metrics to file directory
output_file = "cluster_PD_vs_Control_results.csv"
results_df.to_csv(output_file, index=False, float_format="%.6g")

# Render summary diagnostic logs
print(f"\nReference group: '{ref_group}', Test group: '{test_group}'")
print("Inference quality levels:")
print(" - 'reliable': >=3 samples per condition group")
print(" - 'low_replication': 1-2 samples in at least one group")
print(" - 'undetermined': cluster present in only one condition group\n")
print(f"--- Top reliable clusters ({test_group} vs {ref_group}) ---")

reliable_df = results_df[results_df['inference_quality'] == 'reliable']
if not reliable_df.empty:
    print(reliable_df.to_string(index=False, float_format="%.4f"))
else:
    print("No reliable clusters discovered.")

print(f"\n✅ Analysis complete. Results saved to: {output_file}")

In [ ]:
# Visualize robust cluster enrichment results using a specialized bubble/dot plot layout

from matplotlib.lines import Line2D

# Filter out lower-quality data, keeping only statistically reliable clusters
plot_df = results_df[results_df['inference_quality'] == 'reliable'].copy()

if plot_df.empty:
    print("⚠️ Warning: No 'reliable' clusters found to generate abundance plot.")
else:
    # Define FDR threshold for statistical significance
    alpha = 0.05
    plot_df["significant"] = plot_df["p_adjusted"] < alpha

    def get_color(coef, significant):
        if not significant or pd.isna(coef):
            return "lightgray"
        return "blue" if coef < 0 else "red"

    # Map colors based on direction of change and significance
    plot_df["color"] = plot_df.apply(
        lambda row: get_color(row["coef"], row["significant"]), axis=1
    )

    # Scale bubble sizes non-linearly based on total cell count in each cluster
    n_cells = plot_df["total_cells_in_cluster"]
    min_size, max_size = 100, 450
    if n_cells.max() > n_cells.min():
        sizes = min_size + (max_size - min_size) * ((n_cells - n_cells.min()) / (n_cells.max() - n_cells.min()))**2
    else:
        sizes = np.full(len(n_cells), (min_size + max_size) / 2)

    # Initialize plot layout
    plt.figure(figsize=(12, 6.5), dpi=150)
    x_labels = plot_df["cluster"].astype(str)
    y_values = plot_df["coef"]

    # Render data points
    plt.scatter(
        range(len(x_labels)),
        y_values,
        s=sizes,
        c=plot_df["color"],
        edgecolors="black",
        linewidth=0.5,
        alpha=0.9,
        zorder=3
    )

    # Apply structural plot formatting and gridlines
    plt.axhline(y=0, color="gray", linestyle="-", linewidth=1, alpha=0.7)
    plt.xticks(range(len(x_labels)), x_labels, rotation=45, ha='right')
    plt.xlabel("Cluster ID", fontsize=12)

    # FIXED: Corrected reference direction to match the log odds sign (PD vs Control)
    plt.ylabel("Log Odds Ratio (PD vs Control)", fontsize=12)

    plt.title(
        "Differential Abundance in Dataset 1 (PD vs Control)\n"
        "(Size ∝ total cells; Color = direction & FDR significance)",
        fontsize=13
    )

    # Build custom legend items matching the statistical groups
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="red",
               markersize=10, label=f"Enriched in {test_group}"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="blue",
               markersize=10, label=f"Enriched in {ref_group}"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="lightgray",
               markersize=10, label="Not significant (FDR ≥ 0.05)")
    ]

    plt.legend(handles=legend_elements, loc="upper right", fontsize=10, frameon=True, fancybox=True)
    plt.grid(axis="y", alpha=0.3, linestyle=":")
    plt.tight_layout()
    plt.show()

**Statistical Validation of Disease-Enriched Myeloid Subpopulations**

The quasi-binomial regression framework successfully validates our core biological hypothesis. Notably, **Cluster 1**, which serves as the primary *GPNMB*+ signature hub expressing key activation markers (*MITF*, *PTPRG*, etc.), demonstrates a strong and statistically significant enrichment within the Parkinson's Disease (PD) cohort (Log Odds Ratio > 1.5). This robust alignment between the computational model and our independent literature-derived signatures confirms the pathogenic relevance of this specific microglial state.

# **Secion 9: Pseudobulk analysis**

In this final section, we perform pseudobulk aggregation to enable robust differential gene expression analysis between Parkinson's Disease and Controls. We sum the raw UMI counts across cells for each individual donor to create sample-level profiles, effectively converting single-nucleus data into bulk-like matrices. This aggregation addresses the pseudo-replication issue and allows us to apply the `PyDESeq2` framework for statistical testing.


In [ ]:
! pip install pydeseq2

In [ ]:
# Aggregate single-nucleus raw counts into a sample-level pseudobulk AnnData matrix for PyDESeq2

from scipy.sparse import csr_matrix

# 1. Extract the raw sparse UMI count matrix from layers
counts_sparse = combined.layers['counts']

# Ensure the matrix is in CSR format for efficient row slicing
if not isinstance(counts_sparse, csr_matrix):
    counts_sparse = counts_sparse.tocsr()

# 2. Retrieve biological donor sample labels for each nucleus
samples = combined.obs['sample'].values

# Identify unique donor samples and get index mapping integers
unique_samples, inverse = np.unique(samples, return_inverse=True)
n_samples = len(unique_samples)
n_genes = counts_sparse.shape[1]

# Initialize an empty dense array to store aggregated pseudobulk profiles
pseudobulk_matrix = np.zeros((n_samples, n_genes), dtype=np.float32)

# 3. Sum UMI counts per individual donor across all genes
for i in range(n_samples):
    # Find positions of all cells belonging to the current donor sample
    cell_indices = np.where(inverse == i)[0]
    # Sum raw counts and flatten the matrix block into a 1D array
    pseudobulk_matrix[i] = counts_sparse[cell_indices].sum(axis=0).A1

# 4. Construct sample-level metadata framework
# FIXED: Changed 'condition' column name to 'Clinical diagnosis' to match data structure
required_meta = ['sample', 'Clinical diagnosis', 'Sex']
sample_metadata = combined.obs[required_meta].drop_duplicates().set_index('sample')
sample_metadata = sample_metadata.loc[unique_samples]  # Enforce matching sample ordering

# 5. Create the unified pseudobulk AnnData object
pseudobulk_adata = ad.AnnData(
    X=pseudobulk_matrix,
    obs=sample_metadata,
    var=combined.var.copy()
)

# Force count matrix values to integers as strictly required by PyDESeq2 architectures
pseudobulk_adata.X = pseudobulk_adata.X.astype(np.int64)

# Render structural summary diagnostic logs
print("✅ Pseudobulk AnnData object successfully created:")
print(pseudobulk_adata)

In [ ]:
# Initialize the DESeq2 dataset object using a compatible design factor list

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# 1. Rename the metadata column to eliminate spaces (fixes FormulaSyntaxError)
pseudobulk_adata.obs = pseudobulk_adata.obs.rename(
    columns={"Clinical diagnosis": "Clinical_diagnosis"}
)

# 2. Initialize DeseqDataSet using the list-based design_factors argument
dds = DeseqDataSet(
    adata=pseudobulk_adata,
    design_factors=["Clinical_diagnosis"],  # Pass as a list to match current pydeseq2 API
    refit_cooks=True
)

# 3. Run the complete DESeq2 normalization and fitting pipeline
dds.deseq2()

print("✅ DESeq2 model fitting complete. Dispersion and size factors calculated successfully.")

In [ ]:
# Run the Wald test to compute log2 fold changes and adjusted p-values between conditions

stat_res = DeseqStats(
    dds,
    contrast=["Clinical_diagnosis", "Parkinson's", "Control"]
)

# Render the summary data table of differential expression results
stat_res.summary()

In [ ]:
# Format, clean, and export PyDESeq2 differential expression results to an Excel spreadsheet

import re

# 1. Extract results into a clean pandas DataFrame copy
results_df = stat_res.results_df.copy()

# 2. Extract gene symbols directly from the index (since features are indexed by symbols)
results_df['gene_symbol'] = results_df.index

# Safe check: if 'gene_ids' is missing in pseudobulk_adata.var, populate Ensembl with empty strings
# This prevents KeyError and lets the pipeline finish successfully
if 'gene_ids' in pseudobulk_adata.var.columns:
    results_df['Ensembl'] = results_df['gene_symbol'].map(pseudobulk_adata.var['gene_ids'])
else:
    results_df['Ensembl'] = ""

# Function to remove invalid hidden control characters that break Excel files
def clean_for_excel(s):
    if pd.isna(s):
        return ""
    if isinstance(s, str):
        # Remove control characters (0x00–0x1F, excluding standard whitespaces)
        return re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', '', s)
    return str(s)

# Apply string cleaning to prevent Excel export corruption
results_df['gene_symbol'] = results_df['gene_symbol'].apply(clean_for_excel)

# Fallback mechanism: if gene symbol is empty, replace with Ensembl ID (if available)
results_df['gene_symbol'] = results_df.apply(
    lambda row: row['Ensembl'] if row['gene_symbol'] == "" and row['Ensembl'] != "" else row['gene_symbol'],
    axis=1
)

# 3. Systematically organize and sort columns by statistical significance (FDR)
cols = ['Ensembl', 'gene_symbol', 'baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj']
results_df = results_df[cols].sort_values('padj', na_position='last')

# 4. Save results to the working directory
output_file = "DEG_microglia_Parkinson_vs_Control.xlsx"
results_df.to_excel(output_file, index=False)

# Render summary diagnostic logs
print(f"✅ Differential expression spreadsheet successfully saved to: {output_file}")
print("\nTop 10 differentially expressed genes sorted by padj:")
print(results_df.head(10)[['gene_symbol', 'log2FoldChange', 'padj']])

In [ ]:
# Generate a Volcano plot highlighting significantly differentially expressed genes

# 1. Prepare data structure
df = results_df.copy()

# Compute -log10 adjusted p-values safely
df['log10_padj'] = -np.log10(df['padj'])

# Handle missing and infinite statistical metrics logically
df['log10_padj'] = df['log10_padj'].fillna(0)
max_real_val = df.loc[df['log10_padj'] != np.inf, 'log10_padj'].max() if df['log10_padj'].max() == np.inf else 10
df['log10_padj'] = df['log10_padj'].replace([np.inf, -np.inf], max_real_val + 2)

# 2. Set strict academic thresholds
padj_threshold = 0.05
lfc_threshold = 0.5  # |log2FC| > 0.5 threshold criteria

# Define dot coloring scheme matching expression directions
df['color'] = 'gray'  # Background baseline
df.loc[(df['padj'] < padj_threshold) & (df['log2FoldChange'] > lfc_threshold), 'color'] = 'red'   # Up-regulated in PD
df.loc[(df['padj'] < padj_threshold) & (df['log2FoldChange'] < -lfc_threshold), 'color'] = 'blue'  # Down-regulated in PD

# 3. Construct the scatter plot layout
plt.figure(figsize=(11, 6.5), dpi=300)
plt.scatter(df['log2FoldChange'], df['log10_padj'], c=df['color'], s=12, alpha=0.6, edgecolors='none')

# Add explicit threshold indicator lines
plt.axhline(-np.log10(padj_threshold), color='black', linestyle='--', linewidth=0.8, alpha=0.5)
plt.axvline(lfc_threshold, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
plt.axvline(-lfc_threshold, color='black', linestyle='--', linewidth=0.8, alpha=0.5)

# Configure typography and text blocks
plt.xlabel('Log2 Fold Change (PD vs Control)', fontsize=12)
plt.ylabel(r'$-\log_{10}(\text{adjusted } p\text{-value})$', fontsize=12)
plt.title('Volcano Plot: Microglial Differential Gene Expression (Dataset 1)', fontsize=14, pad=15)

# Construct standard publication legends
plt.scatter([], [], c='red', s=20, label='Up-regulated in PD')
plt.scatter([], [], c='blue', s=20, label='Down-regulated in PD')
plt.scatter([], [], c='gray', s=20, label='Not significant')
plt.legend(loc='upper right', frameon=True, fontsize=10)

# 4. Annotate top 10 genes with native overlap prevention
top_genes = df[df['padj'] < padj_threshold].sort_values('padj').head(10)

# We use alternating offsets to prevent horizontal stacking
for idx, (_, row) in enumerate(top_genes.reset_index().iterrows()):
    x = row['log2FoldChange']
    y = row['log10_padj']

    # Smart horizontal shift: away from the center axis to separate overlapping labels
    x_offset = -0.15 if x < 0 else 0.15
    # Zig-zag vertical shift: alternates between 0.1 and 0.4 based on item index
    y_offset = 0.1 if idx % 2 == 0 else 0.4

    plt.text(
        x + x_offset,
        y + y_offset,
        row['gene_symbol'],
        fontsize=9,
        fontstyle='italic',
        weight='bold',
        ha='right' if x < 0 else 'left',  # Align text away from the points
        va='bottom'
    )

# Force a generous vertical safety margin at the top so labels never clip out of the border
plt.ylim(bottom=-0.5, top=df['log10_padj'].max() + 1.5)

# Save high-resolution visualization block to memory
plt.tight_layout()
plt.savefig("volcano_plot_PD_microglia.png", dpi=300)
plt.show()

print("✅ Volcano plot successfully generated and exported as volcano_plot_PD_microglia.png")